# Projet DPE Paris : étiquettes énergétiques des logements existants
**MSA-DATI07-03 · Advanced EDA & Data Collection** · Groupe : Elie Maarek, Joshai Bensoussan, Lucas Noury

On collecte des vrais DPE (diagnostics de performance énergétique) via l'API de l'ADEME, on vérifie la qualité des données, puis on les prépare pour la suite du projet (prédire la classe DPE d'un logement).

| # | Partie | Question |
|---|---|---|
| 1 | Contexte & provenance | D'où viennent les données et pour quelle question ? |
| 2 | Paramètres | Quelqu'un d'autre peut-il relancer le notebook et obtenir la même chose ? |
| 3 | Collecte & schéma | Les données sont-elles bien collectées et conformes à ce qu'on pense ? |
| 4 | Qualité des données | Qu'est-ce qui manque, est en double ou bizarre, et qu'est-ce qu'on décide ? |
| 5 | Exploration | Que disent les données ? Chaque graphique a une légende. |
| 6 | Résultats | À quoi peut-on se fier et que faut-il tester ensuite ? |

**Livrables :** ce notebook (Restart & Run All, même fingerprint deux fois), `requirements.txt`, `data/raw/` (parquet + README), le rapport qualité (2 pages), l'historique Git.

---
## 1 · Context & provenance

| Field | Your answer |
|---|---|
| **Business question** | Comment les étiquettes DPE (A à G) sont réparties à Paris, et comment elles varient selon la période de construction, la surface et le type de chauffage ? À terme : savoir quelles variables permettent de prédire la classe DPE d'un logement. |
| **Source** | ADEME open-data portal (data.ademe.fr), data-fair API, dataset `dpe03existant` |
| **Dataset page / API docs** | https://data.ademe.fr/datasets/dpe03existant |
| **Collected by / when** | Elie Maarek, Joshai Bensoussan et Lucas Noury, collecte du 5 novembre 2026 (heure exacte dans `data/raw/README.md`) |
| **Collection method** | REST API, paginated, cached with `requests-cache` (see Section 3) |
| **Filters applied** | `code_departement_ban:"75"` (Paris), `sort=_rand` (tirage aléatoire), pages de 1000 lignes, 10 000 lignes maximum (`MAX_ROWS`). Le jeu `dpe03existant` exclut déjà les DPE désactivés. |
| **Licence / constraints** | Licence Ouverte 2.0 (Etalab), source à citer : ADEME |
| **Unit of observation** | Une ligne = un DPE (identifiant `numero_dpe`). Ce n'est pas toujours un logement entier : la colonne `methode_application_dpe` mélange des DPE de logement, d'immeuble et des appartements générés à partir du DPE de l'immeuble. Un même logement peut aussi avoir eu plusieurs DPE (`numero_dpe_remplace`). Vérification chiffrée dans la cellule sous le tableau `overview`. |
| **Known limitations** | On a 10 000 lignes sur 862 377 pour Paris (environ 1,2 %), tirées au hasard à un instant donné : le jeu est mis à jour chaque semaine, donc une nouvelle collecte donne d'autres lignes. Un DPE n'est obligatoire que pour une vente, une location ou une construction neuve, l'échantillon n'est donc pas représentatif de tout le parc. Les données sont brutes (saisies par les diagnostiqueurs, l'ADEME ne les corrige pas). L'API renvoie 223 colonnes, dont beaucoup sont presque vides. L'étiquette DPE est calculée à partir de la consommation d'énergie primaire et des émissions de GES (double seuil depuis juillet 2021), ces variables ne devront pas servir à la prédire. Paris uniquement. |

---
## 2 · Setup & parameters
Paramètres, imports et `requirements.txt`. Pour installer l'environnement : `pip install -r requirements.txt`.

In [44]:
# --- Parameters: the ONLY cell you should need to edit to rerun on new data ---
SEED = 42
DATA_DIR = "../data"
REPORT_DIR = "../reports"
CACHE_HOURS = 24 * 7                              # re-use API responses for a week
IQR_K = 1.5
MAX_MISSING_SHARE = 0.30

BASE_URL = "https://data.ademe.fr/data-fair/api/v1/datasets/dpe03existant/lines"
QUERY = {"size": 1000, "sort": "_rand",
         "qs": 'code_departement_ban:"75"'}      # Elasticsearch-style filter; if 0 rows come back, check the field name in the API docs
MAX_ROWS = 10_000
PAUSE_S = 0.5

In [45]:
import hashlib, json, random, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pandera
import pandera.pandas as pa
import requests
import requests_cache
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

random.seed(SEED); np.random.seed(SEED)
DATA, REPORTS = Path(DATA_DIR), Path(REPORT_DIR)
for p in (DATA / "raw", DATA / "processed", REPORTS):
    p.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

for lib in (pd, np, sns, pandera, requests):
    print(f"{lib.__name__:<10}{lib.__version__}")

pandas    3.1.1
numpy     2.5.4
seaborn   0.13.2
pandera   0.35.0
requests  2.34.2


In [46]:
# Write requirements.txt from the packages this notebook imports
from importlib.metadata import version
pkgs = ["pandas", "numpy", "matplotlib", "seaborn", "pandera", "pyarrow", "requests", "requests-cache"]
Path("../requirements.txt").write_text("".join(f"{p}=={version(p)}\n" for p in pkgs))
print(Path("../requirements.txt").read_text())

pandas==3.1.1
numpy==2.5.4
matplotlib==3.11.2
seaborn==0.13.2
pandera==0.35.0
pyarrow==26.0.1
requests==2.34.2
requests-cache==1.3.3



---
## 3 · Collection, ingestion & schema

### 3.1 A polite, cached HTTP session
- **Cache**: the same request is never sent twice (fast reruns, kind to the API).
- **Retry with backoff** on 429 / 5xx, respecting `Retry-After`.
- **Throttle**: pause between requests that actually hit the network.

In [47]:
session = requests_cache.CachedSession("cache/api", expire_after=CACHE_HOURS * 3600)
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
    respect_retry_after_header=True)))
session.headers["User-Agent"] = "MSA-DATI07-student-project/1.0"

def get(url, params=None):
    r = session.get(url, params=params, timeout=30)
    r.raise_for_status()
    if not getattr(r, "from_cache", False):
        time.sleep(PAUSE_S)
    return r

### 3.2 Collect

In [48]:
def collect_next_link(url, params, items_key, max_rows=MAX_ROWS):
    """APIs that return the URL of the next page in the response ("next").
    We follow it until it disappears or we have enough rows."""
    rows, first = [], True
    while url and len(rows) < max_rows:
        r = get(url, params=params if first else None)   # the next link already contains the params
        page = r.json()
        if first:
            total = page.get("total", page.get("count"))
            print(f"API reports {total:,} matching rows" if total is not None else "total unknown")
        rows += page.get(items_key, [])
        url, first = page.get("next"), False
    return rows[:max_rows]

In [49]:
collected_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
raw_rows = collect_next_link(BASE_URL, QUERY, items_key="results")
print(f"collected {len(raw_rows):,} rows at {collected_at}")

API reports 862,377 matching rows
collected 10,000 rows at 2026-11-05T18:24:29+00:00


### 3.3 Flatten, save raw Parquet + README

In [50]:
df_raw = pd.json_normalize(raw_rows, sep='_')

In [51]:
raw_path = DATA / "raw" / "raw.parquet"
df_raw.to_parquet(raw_path, index=False)
(DATA / "raw" / "README.md").write_text(f"""# Raw data
- Source: see Section 1 of the notebook
- Collected at (UTC): {collected_at}
- Rows x columns: {df_raw.shape[0]} x {df_raw.shape[1]}
- Columns: {", ".join(df_raw.columns)}
""")
print(df_raw.shape)
df_raw.head()

(10000, 223)


,configuration_installation_chauffage_n1,conso_chauffage_installation_chauffage_n1,type_generateur_n1_ecs_n1,numero_voie_ban,score_ban,surface_habitable_immeuble,conso_auxiliaires_ep,deperditions_murs,cout_eclairage,conso_auxiliaires_ef,...,type_generateur_n2_installation_n2,type_energie_generateur_n2_installation_n2,conso_chauffage_generateur_n2_installation_n2,facteur_couverture_solaire_saisi_installation_chauffage_n1,type_generateur_n2_ecs_n1,volume_stockage_generateur_n2_ecs_n1,usage_generateur_n2_ecs_n1,conso_ef_generateur_n2_ecs_n1,type_energie_generateur_n2_ecs_n1,description_generateur_n2_ecs_n1
0,Installation de chauffage simple,204265.3,Chaudière gaz à condensation 2001-2015,12,0.65,4084.0,620.9,727.3,37.8,270.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Installation de chauffage simple,26185.0,Réseau de chaleur isolé,10,0.75,1780.0,814.4,178.8,111.9,354.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Installation de chauffage simple,2709.4,Chaudière gaz basse température 2001-2015,47,0.89,865.0,371.9,15.4,14.0,161.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Installation de chauffage simple,1303.8,Chaudière gaz basse température après 2015,51b,0.80,11.0,411.7,7.5,6.0,179.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Installation de chauffage simple,1.0,Ballon électrique à accumulation vertical Caté...,70,0.77,NaN,628.1,37.2,11.3,273.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
# First look: types, missing share, number of distinct values per column
overview = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "share_missing": df_raw.isna().mean().round(3),
    "n_unique": df_raw.nunique(dropna=True),
}).sort_values("share_missing", ascending=False)
overview

,dtype,share_missing,n_unique
description_generateur_n2_ecs_n1,str,1.0,1
usage_generateur_n2_installation_n2,str,1.0,2
_score,object,1.0,0
type_energie_generateur_n2_ecs_n1,str,1.0,1
facteur_couverture_solaire_installation_chauffage_n1,float64,1.0,2
...,...,...,...
emission_ges_5_usages,float64,0.0,8322
date_derniere_modification_dpe,str,0.0,289
etiquette_ges,str,0.0,7
_id,str,0.0,10000


### 3.3 bis · Vérifs rapides (unité d'observation)

In [ ]:
# vérif : une ligne = un DPE ? et quels types de lignes on a
print("numero_dpe unique :", df_raw["numero_dpe"].is_unique)
print(df_raw["methode_application_dpe"].value_counts(dropna=False))
print(df_raw["type_batiment"].value_counts(dropna=False))
print(df_raw["etiquette_dpe"].value_counts().sort_index())
print(df_raw["surface_habitable_logement"].describe())
print(df_raw["conso_5_usages_par_m2_ep"].describe())
n_remplace = df_raw["numero_dpe_remplace"].notna().sum() if "numero_dpe_remplace" in df_raw else 0
print(n_remplace, "DPE avec un ancien DPE remplacé")

### 3.4 Schema-as-contract

*TODO (Exercise 3 of Session 6)*: write a `pa.DataFrameSchema` for the columns you keep. Requirements: every kept column with a dtype, `strict=True`, **at least 4 domain checks**, `nullable=True` only where justified in a comment. Validate the raw data with `lazy=True` and keep the failure summary for your report.

Columns to look at first (confirm the exact names in `df_raw.columns`): 200+ columns. Start with: the DPE number, `etiquette_dpe`, `etiquette_ges`, `date_etablissement_dpe`, `surface_habitable_logement`, `periode_construction`, `type_energie_principale_chauffage`, `conso_5_usages_par_m2_ep`, `code_postal_ban` (names TO CONFIRM).

In [53]:
# TODO: select the columns you keep, then write the contract
KEEP = list(df_raw.columns)          # TODO: replace by your selection (and log it as a decision)
df_sel = df_raw[KEEP].copy()

schema = None                        # TODO: replace None by pa.DataFrameSchema({...}, strict=True)

if schema is not None:
    try:
        schema.validate(df_sel, lazy=True)
        print("✅ raw data passes the contract")
    except pa.errors.SchemaErrors as err:
        fails = err.failure_cases
        print(f"❌ {len(fails)} violations")
        display(fails.groupby(["column", "check"]).size().rename("n").reset_index())

---
## 4 · Data quality — measure → decide → log

Every decision gets one `log(...)` call. The table at the end of this section goes into your report.

**What to look for in this dataset:**
- Too many columns: select 15–25 that serve your question and **log the selection** as a decision.
- Missingness per column will range from 0% to ~100%: which columns are usable at all?
- Impossible values: surfaces of 0 or 5,000 m², consumptions per m² far beyond the G threshold.
- Several DPE for the same address: re-diagnosis, or duplicates? Decide which one you keep.

In [54]:
decisions = []
def log(column, issue, n_rows, decision, why):
    decisions.append(dict(column=column, issue=issue, n_rows=n_rows, decision=decision, why=why))

df = df_sel.copy()

### 4.1 Duplicates

In [55]:
n_exact = int(df.astype(str).duplicated().sum())    # astype(str): some columns may hold dicts
print("exact duplicate rows:", n_exact)
# TODO: decide, apply, and log. Then look for near-duplicates (same entity, different id or spelling).

exact duplicate rows: 0


### 4.2 Impossible values

In [56]:
# TODO: for each numeric column, what is physically or logically impossible? Count, decide, log.
df.describe().T

,count,mean,std,min,25%,50%,75%,max
conso_chauffage_installation_chauffage_n1,9845.0,203547.185719,606830.257633,1.0,3206.100,6960.00,24879.500,5188223.80
score_ban,10000.0,0.749294,0.104980,0.0,0.690,0.75,0.800,0.99
surface_habitable_immeuble,5815.0,5745.157885,17963.509367,1.0,1335.000,2948.00,6522.000,1000000.00
conso_auxiliaires_ep,10000.0,825.183060,4797.320287,0.0,23.975,409.10,752.325,295925.60
deperditions_murs,9977.0,1319.372607,3646.744035,0.0,27.300,67.70,619.400,30544.10
...,...,...,...,...,...,...,...,...
facteur_couverture_solaire_installation_chauffage_n1,3.0,0.233333,0.115470,0.1,0.200,0.30,0.300,0.30
conso_chauffage_generateur_n2_installation_n2,3.0,205.566667,356.051911,0.0,0.000,0.00,308.350,616.70
facteur_couverture_solaire_saisi_installation_chauffage_n1,1.0,0.100000,NaN,0.1,0.100,0.10,0.100,0.10
volume_stockage_generateur_n2_ecs_n1,1.0,200.000000,NaN,200.0,200.000,200.00,200.000,200.00


### 4.3 Missingness — mechanism before method

In [57]:
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0].rename("share_missing").to_frame()
miss["flag"] = np.where(miss.share_missing > MAX_MISSING_SHARE, "⚠️ high", "")
miss

,share_missing,flag
_score,1.0000,⚠️ high
description_generateur_n2_ecs_n1,0.9999,⚠️ high
type_energie_generateur_n2_ecs_n1,0.9999,⚠️ high
conso_ef_generateur_n2_ecs_n1,0.9999,⚠️ high
usage_generateur_n2_ecs_n1,0.9999,⚠️ high
...,...,...
zone_climatique,0.0021,
qualite_isolation_murs,0.0004,
adresse_ban,0.0002,
identifiant_ban,0.0002,


In [ ]:
# TODO: for the 2–3 most important columns with missing values, test whether missingness
# depends on another observed column (MAR) — e.g.
# df.assign(m=df["<col>"].isna()).groupby("<other col>")["m"].mean()
# Then name the mechanism (MCAR / MAR / MNAR), decide, log.

### 4.4 Outliers — triage, don't delete

In [ ]:
# TODO: pick your key numeric column. Use an IQR fence (on a log scale if it is skewed),
# look at the flagged rows, separate errors from real extremes, decide per group, log.

In [58]:
decision_table = pd.DataFrame(decisions, columns=["column", "issue", "n_rows", "decision", "why"])
decision_table.to_csv(REPORTS / "decision_table.csv", index=False)
decision_table

,column,issue,n_rows,decision,why


---
## 5 · Exploration — every figure gets a caption and a *so what*

In [59]:
def show(fig, title, caption, so_what, name):
    fig.suptitle(title, x=0.01, ha="left", fontsize=12, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    fig.text(0.01, -0.04, f"{caption}\n→ {so_what}", ha="left", va="top", fontsize=9, wrap=True)
    fig.savefig(REPORTS / f"{name}.png", bbox_inches="tight", dpi=120)
    plt.show()

clean = df.copy()      # TODO: this should be your cleaned data after Section 4

### 5.1 Univariate

In [ ]:
# TODO: one figure showing the distribution of your key variable (ECDF before `.describe()`)
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_1")

### 5.2 Bivariate

In [ ]:
# TODO: one figure showing your key variable against 1–2 explanatory variables
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_2")

### 5.3 Multivariate

In [ ]:
# TODO: one figure showing the correlation structure, or a breakdown by two categories
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_3")

---
## 6 · Findings, what cannot be trusted, next steps

*TODO*

**Three findings** — each with the figure it comes from and what it means for modelling.
1.
2.
3.

**What cannot be trusted** — column, subset, and why.
-
-

**Five testable hypotheses** — variable, direction, population, test.
- H1:
- H2:
- H3:
- H4:
- H5:

### Final contract & fingerprint
*TODO*: write `clean_schema` for the data a modeller receives (it may differ from the raw schema). Then **Restart & Run All twice**: the fingerprint must be identical.

In [60]:
clean_schema = None                  # TODO: pa.DataFrameSchema({...}, strict=True)
if clean_schema is not None:
    clean_schema.validate(clean, lazy=True)
    print("✅ clean data passes the contract")

clean.to_parquet(DATA / "processed" / "clean.parquet", index=False)
fingerprint = hashlib.sha256(
    pd.util.hash_pandas_object(clean.astype(str), index=False).values.tobytes()).hexdigest()[:12]
print("rows:", len(clean), "· fingerprint:", fingerprint)

rows: 10000 · fingerprint: 20922ad835d0


> **Reproducibility note.** The fingerprint is stable between runs *thanks to the cache*: the API data changes over time, so a run without the cache (or after `CACHE_HOURS`) can give a different fingerprint. That is expected — record the collection date in Section 1 and in the README, and commit the raw Parquet if it is small enough.